# Retrieval-Augmented Generation (RAG) from Scratch

## Learning objectives

By the end of this notebook, you should be able to:

- Explain what Retrieval-Augmented Generation, or RAG, means.
- Describe why LLMs alone may hallucinate or answer from stale knowledge.
- Explain how retrieval helps ground an answer in provided information.
- Identify the key components of a RAG system: documents, chunks, embeddings, vector search, context, and generation.
- Understand the difference between "the model knows something" and "the model is given relevant context".
- Build a small RAG pipeline using the official OpenAI Python SDK, embeddings, cosine similarity, and an LLM call.

This notebook does not use LangChain or a vector database. Everything is kept in memory so the full pipeline is visible.

## 1. What is RAG?

RAG stands for **Retrieval-Augmented Generation**.

It combines two steps:

- **Retrieval**: find relevant information from an external knowledge base.
- **Generation**: ask the LLM to answer using the retrieved context.

A simple RAG flow looks like this:

```text
User question
   |
   v
Retrieve relevant chunks
   |
   v
Add chunks to prompt
   |
   v
LLM generates grounded answer
```

The important idea is that the LLM is not only using what it learned during training. We give it relevant information at question time.

## 2. Why do we need RAG?

LLMs are powerful, but they have limits:

- Their knowledge can be limited or stale.
- They may produce answers that sound confident but are incorrect. This is called **hallucination**.
- They usually do not know private company documents, internal policies, or project-specific information.
- Organizations often need answers grounded in approved internal documents.

RAG can reduce hallucinations by giving the model relevant source material. However, RAG does not guarantee truth. The answer still depends on document quality, retrieval quality, prompt quality, and the model's behavior.

## 3. Key concepts

- **Document**: an original piece of knowledge, such as a policy, FAQ, manual, contract, or wiki page.
- **Chunk**: a smaller piece of a document. RAG systems usually search chunks instead of entire documents.
- **Embedding**: a list of numbers that represents the meaning of text.
- **Vector database / vector index**: a system that stores embeddings and searches for similar vectors. In this notebook, we use a simple Python list instead.
- **Similarity search**: finding chunks whose embeddings are closest to the query embedding.
- **Context window**: the maximum amount of text the model can receive in one request.
- **Grounded answer**: an answer based on the provided context instead of unsupported guesses.
- **Citation / source reference**: a note showing which document or chunk supported the answer.

## 4. Setup

We will use:

- `openai` for embeddings and final answer generation.
- `numpy` for vector math.
- `OPENAI_API_KEY` from the environment.

Before running this cell, make sure your API key is available as an environment variable named `OPENAI_API_KEY`.

In [2]:
import os
import importlib

import numpy as np
import utils.open_ai as _openai_module
importlib.reload(_openai_module)
from utils.open_ai import OpenAI

EMBEDDING_MODEL = 'text-embedding-3-small'
GENERATION_MODEL = 'gpt-4.1-mini'

client = OpenAI()

print('Setup complete.')


Setup complete.


## 5. Toy knowledge base

Real RAG systems usually read from PDFs, web pages, databases, SharePoint, Confluence, or other document sources.

For teaching, we will keep the knowledge base directly inside the notebook. We will use fictional company policies so there are no external files or private dependencies.

In [15]:
documents = [
    {
        'id': 'vacation_policy',
        'title': 'Company Vacation Policy',
        'text': '''Full-time employees receive 20 paid vacation days per calendar year. Vacation requests should be submitted at least 10 business days before the planned time off.

Managers should approve or reject vacation requests within 5 business days. Unused vacation days do not carry over into the next calendar year.

Contractors do not receive paid vacation days through the company. Contractors should follow the terms of their individual contract for time away from work.'''
    },
    {
        'id': 'laptop_replacement_policy',
        'title': 'Laptop Replacement Policy',
        'text': '''Employees are eligible for a standard laptop replacement every 3 years. The replacement cycle is based on the device assignment date in the IT asset system.

Early replacement may be approved if the laptop is damaged, lost, or no longer supports required work. Early replacement requires manager approval and IT review.

Accessories such as keyboards, mice, and laptop stands can be requested separately through the IT service portal.'''
    },
    {
        'id': 'security_policy',
        'title': 'Security Policy',
        'text': '''Confidential client data must not be uploaded to public AI tools, public file-sharing websites, or unapproved external services.

Employees must use approved company systems when working with confidential or regulated data. Security incidents or accidental data exposure must be reported to the Security team immediately.

Passwords must never be shared in chat, email, documents, support tickets, or AI prompts.'''
    },
    {
        'id': 'travel_reimbursement_policy',
        'title': 'Travel Reimbursement Policy',
        'text': '''Employees can be reimbursed for approved business travel expenses. Eligible expenses include flights, hotels, conference registration, and ground transportation.

Taxi or rideshare trips from the airport to a hotel, office, or client site are reimbursable when the trip is for approved business travel. A receipt is required.

Meal reimbursement is capped at 75 USD per day unless a client contract states a lower amount. Personal travel, entertainment, and upgrades are not reimbursable.

Expense reports must be submitted within 30 days after the trip ends.'''
    },
    {
        'id': 'internal_ai_assistant_policy',
        'title': 'Internal AI Assistant Policy',
        'text': '''The internal AI assistant may be used to summarize internal documents, draft non-client communications, brainstorm ideas, and explain technical concepts.

The internal AI assistant may process confidential data only when the tool is approved for that data classification and the user has permission to access the data.

AI-generated outputs must be reviewed by a human before being used for client deliverables, legal decisions, financial decisions, or security decisions.'''
    }
]

print(f'Loaded {len(documents)} documents:')
for doc in documents:
    print(f"- {doc['title']}")

Loaded 5 documents:
- Company Vacation Policy
- Laptop Replacement Policy
- Security Policy
- Travel Reimbursement Policy
- Internal AI Assistant Policy


## 6. Chunking

Instead of embedding a whole document, we split each document into smaller chunks.

Why chunk?

- Smaller chunks make retrieval more precise.
- Large documents may contain many unrelated topics.
- The LLM has a limited context window.

Our chunking strategy will be intentionally simple: split by paragraphs and group a few paragraphs together. Real systems often need better chunking based on document structure, headings, tables, code blocks, page numbers, and metadata.

In [4]:
def chunk_text(text: str, max_paragraphs: int = 2) -> list[str]:
    """Split text into small paragraph-based chunks."""
    paragraphs = [paragraph.strip() for paragraph in text.split('\n\n') if paragraph.strip()]
    chunks = []

    for start in range(0, len(paragraphs), max_paragraphs):
        group = paragraphs[start:start + max_paragraphs]
        chunks.append('\n\n'.join(group))

    return chunks


def build_chunks_from_documents(documents: list[dict], max_paragraphs: int = 2) -> list[dict]:
    """Create searchable chunks and keep useful source metadata."""
    all_chunks = []

    for doc in documents:
        for index, text in enumerate(chunk_text(doc['text'], max_paragraphs=max_paragraphs), start=1):
            all_chunks.append({
                'chunk_id': f"{doc['id']}_chunk_{index}",
                'doc_id': doc['id'],
                'title': doc['title'],
                'text': text,
            })

    return all_chunks


chunks = build_chunks_from_documents(documents)

print(f'Created {len(chunks)} chunks.\n')
for chunk in chunks[:3]:
    print(chunk['chunk_id'], '|', chunk['title'])
    print(chunk['text'][:220] + '...')
    print()

Created 10 chunks.

vacation_policy_chunk_1 | Company Vacation Policy
Full-time employees receive 20 paid vacation days per calendar year. Vacation requests should be submitted at least 10 business days before the planned time off.

Managers should approve or reject vacation requests withi...

vacation_policy_chunk_2 | Company Vacation Policy
Contractors do not receive paid vacation days through the company. Contractors should follow the terms of their individual contract for time away from work....

laptop_replacement_policy_chunk_1 | Laptop Replacement Policy
Employees are eligible for a standard laptop replacement every 3 years. The replacement cycle is based on the device assignment date in the IT asset system.

Early replacement may be approved if the laptop is damaged, lo...



## 7. Embedding generation

An embedding converts text into a vector: a list of numbers that represents meaning.

Texts with similar meanings should have vectors that are close together. We will embed every chunk once, then embed each user question when we search.

In production, embeddings are usually stored in a vector database. Here, we store them directly in memory using `numpy` arrays.

In [5]:
def get_embedding(text: str) -> list[float]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding

for chunk in chunks:
    chunk['embedding'] = np.array(get_embedding(chunk['text']), dtype=np.float32)

print(f'Generated embeddings for {len(chunks)} chunks.')
print(f"Embedding size: {len(chunks[0]['embedding'])}")


Generated embeddings for 10 chunks.
Embedding size: 3072


## 8. Similarity search

Now we need a way to find the chunks most similar to a user question.

We will:

1. Embed the question.
2. Compare the question embedding with each chunk embedding.
3. Sort chunks by similarity score.
4. Return the top results.

We will use **cosine similarity**, a common way to compare vectors by direction.

In [6]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


def retrieve(query: str, top_k: int = 3) -> list[dict]:
    query_embedding = np.array(get_embedding(query), dtype=np.float32)
    scored_chunks = []

    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk['embedding'])
        scored_chunks.append({**chunk, 'score': score})

    scored_chunks.sort(key=lambda item: item['score'], reverse=True)
    return scored_chunks[:top_k]


def print_retrieved(results: list[dict]) -> None:
    for index, chunk in enumerate(results, start=1):
        print(f"\nResult {index} | score: {chunk['score']:.3f} | source: {chunk['title']}")
        print('-' * 80)
        print(chunk['text'])

## 9. First retrieval demo

Before we ask the LLM to answer, let's inspect retrieval by itself.

The question is about airport taxi reimbursement. We expect the travel reimbursement policy to appear near the top.

In [7]:
question = 'Can I get reimbursed for a taxi from the airport?'

retrieved_chunks = retrieve(question, top_k=3)
print_retrieved(retrieved_chunks)


Result 1 | score: 0.657 | source: Travel Reimbursement Policy
--------------------------------------------------------------------------------
Employees can be reimbursed for approved business travel expenses. Eligible expenses include flights, hotels, conference registration, and ground transportation.

Taxi or rideshare trips from the airport to a hotel, office, or client site are reimbursable when the trip is for approved business travel. A receipt is required.

Result 2 | score: 0.430 | source: Travel Reimbursement Policy
--------------------------------------------------------------------------------
Meal reimbursement is capped at 75 USD per day unless a client contract states a lower amount. Personal travel, entertainment, and upgrades are not reimbursable.

Expense reports must be submitted within 30 days after the trip ends.

Result 3 | score: 0.181 | source: Laptop Replacement Policy
--------------------------------------------------------------------------------
Accessories

## 10. Generation without RAG

First, let's ask the LLM the same question without giving it our company policy.

The answer may sound reasonable, but it is not grounded in the policy documents. It may say something generic like "it depends on the company policy" or it may guess.

In [8]:
def ask_llm_without_rag(question: str) -> str:
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user', 'content': question},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


print(ask_llm_without_rag(question))

Whether you can get reimbursed for a taxi from the airport depends on your organization's travel policy or the specific circumstances of your trip. Here are some general guidelines:

1. **Company Travel Policy:**  
   - Check your company’s travel or expense reimbursement policy. Some companies allow reimbursement for taxis from the airport if it is the most reasonable or only option.  
   - Others may require you to use public transportation, ride-sharing services, or pre-approved shuttle services instead.

2. **Pre-Approval:**  
   - Some organizations require pre-approval for taxi expenses. If you did not get approval beforehand, reimbursement might be denied.

3. **Reasonableness:**  
   - If a taxi was necessary due to late arrival times, lack of public transport, or carrying heavy luggage, reimbursement is more likely.

4. **Documentation:**  
   - Keep the taxi receipt as proof of the expense. Without a receipt, reimbursement is usually not possible.

5. **Alternative Options:**

## 11. Generation with RAG

Now we combine retrieval and generation.

The prompt will tell the model:

- Answer only using the provided context.
- If the answer is not in the context, say: `I don't know based on the provided documents.`
- Mention the source title or titles used.

This does not make the system perfect, but it gives the model a much stronger instruction to stay grounded.

In [9]:
def build_rag_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context_blocks = []

    for index, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(
            f"Source {index}: {chunk['title']}\n{chunk['text']}"
        )

    context = '\n\n'.join(context_blocks)

    return f'''You are answering questions for employees using only the provided company policy context.

Rules:
- Answer only using the provided context.
- If the answer is not in the context, say: "I don't know based on the provided documents."
- Mention the source title(s) you used.
- Be concise and beginner-friendly.

Context:
{context}

Question:
{question}

Answer:'''


def generate_answer(prompt: str) -> str:
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content


def answer_with_rag(question: str, top_k: int = 3) -> dict:
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_rag_prompt(question, retrieved)
    answer = generate_answer(prompt)

    print('Question:')
    print(question)
    print('\nAnswer:')
    print(answer)
    print('\nSources retrieved:')
    for chunk in retrieved:
        print(f"- {chunk['title']} (score: {chunk['score']:.3f})")

    return {
        'question': question,
        'answer': answer,
        'retrieved_chunks': retrieved,
    }

## 12. RAG demo questions

Let's try several questions.

The final question asks about gym memberships, which are not mentioned in the documents. A grounded RAG answer should say that it does not know based on the provided documents.

In [10]:
demo_questions = [
    'Can I get reimbursed for a taxi from the airport?',
    'When can I replace my laptop?',
    'Can I upload confidential client data to public AI tools?',
    'How many vacation days do contractors get?',
    'Does the company provide free gym memberships?',
]

for demo_question in demo_questions:
    print('=' * 100)
    answer_with_rag(demo_question, top_k=3)
    print()

Question:
Can I get reimbursed for a taxi from the airport?

Answer:
Yes, you can be reimbursed for a taxi from the airport to a hotel, office, or client site if the trip is for approved business travel. A receipt is required.  
(Source: Travel Reimbursement Policy)

Sources retrieved:
- Travel Reimbursement Policy (score: 0.657)
- Travel Reimbursement Policy (score: 0.430)
- Laptop Replacement Policy (score: 0.181)

Question:
When can I replace my laptop?

Answer:
You can replace your laptop every 3 years based on the device assignment date in the IT asset system. Early replacement is possible if your laptop is damaged, lost, or no longer supports your work, but this requires manager approval and IT review.  
(Source: Laptop Replacement Policy)

Sources retrieved:
- Laptop Replacement Policy (score: 0.602)
- Laptop Replacement Policy (score: 0.257)
- Travel Reimbursement Policy (score: 0.133)

Question:
Can I upload confidential client data to public AI tools?

Answer:
No, you cannot 

## 13. What can go wrong?

RAG is useful, but it is not magic. Common problems include:

- **Bad chunking**: the answer is split away from the important context.
- **Bad retrieval**: the search returns irrelevant chunks.
- **Missing documents**: the knowledge base does not contain the answer.
- **Conflicting documents**: two sources say different things.
- **Context too long**: too much retrieved text makes the prompt noisy or expensive.
- **Model ignores instructions**: the model may still answer beyond the context.
- **Relevant but insufficient text**: the retrieved chunk is related but does not fully answer the question.

RAG reduces hallucination risk, but it does not guarantee truth. Production systems need monitoring and evaluation.

## 14. Improving RAG

Common ways to improve RAG systems include:

- **Better chunking**: split documents using headings, sections, page numbers, or semantic boundaries.
- **Metadata filtering**: search only documents from a department, date range, region, client, or permission group.
- **Hybrid search**: combine keyword search with embedding search.
- **Reranking**: retrieve many chunks, then use another model to rank the best ones.
- **Query rewriting**: rewrite vague user questions into clearer search queries.
- **Citations**: show the exact source titles or passages used.
- **Evaluation datasets**: test the system on known questions and expected answers.
- **Human feedback**: collect user feedback and improve documents, chunking, prompts, or retrieval.

## 15. Mini exercises

Try these exercises during or after class.

### Exercise 1

Change `top_k` from `1` to `3` for a query and compare the answers.

Questions to discuss:

- What changes?
- Does more context always help?

### Solution Exercise 1 - HIDE THIS CELL

This solution compares a narrow retrieval setting with a wider retrieval setting.

In [11]:
# SOLUTION CELL - HIDE BEFORE CLASS

exercise_question = 'Can I get reimbursed for a taxi from the airport?'

print('Answer with top_k=1')
print('=' * 80)
answer_with_rag(exercise_question, top_k=1)

print('\n\nAnswer with top_k=3')
print('=' * 80)
answer_with_rag(exercise_question, top_k=3)

print('\nDiscussion: More context can help if it adds useful evidence, but it can hurt if it adds irrelevant or conflicting text.')

Answer with top_k=1
Question:
Can I get reimbursed for a taxi from the airport?

Answer:
Yes, you can be reimbursed for a taxi from the airport when it is for approved business travel. Make sure to keep the receipt.  
(Source: Travel Reimbursement Policy)

Sources retrieved:
- Travel Reimbursement Policy (score: 0.657)


Answer with top_k=3
Question:
Can I get reimbursed for a taxi from the airport?

Answer:
Yes, you can be reimbursed for a taxi from the airport to a hotel, office, or client site if the trip is for approved business travel. A receipt is required.  
(Source: Travel Reimbursement Policy)

Sources retrieved:
- Travel Reimbursement Policy (score: 0.657)
- Travel Reimbursement Policy (score: 0.430)
- Laptop Replacement Policy (score: 0.181)

Discussion: More context can help if it adds useful evidence, but it can hurt if it adds irrelevant or conflicting text.


### Exercise 2

Add a new document to the knowledge base about remote work policy.

Then rebuild chunks and embeddings. Ask a question about remote work and verify that RAG can answer it.

### Solution Exercise 2 - HIDE THIS CELL

This solution adds a remote work policy, then rebuilds the in-memory index.

In [12]:
# SOLUTION CELL - HIDE BEFORE CLASS

remote_work_document = {
    'id': 'remote_work_policy',
    'title': 'Remote Work Policy',
    'text': '''Employees may work remotely up to 3 days per week if their role allows remote work and their manager approves the arrangement.

Remote employees must be reachable during core collaboration hours from 10:00 to 16:00 local time.

Employees who handle confidential data must use the company VPN and approved devices when working remotely.'''
}

if not any(doc['id'] == remote_work_document['id'] for doc in documents):
    documents.append(remote_work_document)

chunks = build_chunks_from_documents(documents)

for chunk in chunks:
    chunk['embedding'] = np.array(get_embedding(chunk['text']), dtype=np.float32)

print(f'Rebuilt the index with {len(documents)} documents and {len(chunks)} chunks.')
answer_with_rag('Can employees work remotely three days per week?', top_k=3)

Rebuilt the index with 6 documents and 12 chunks.
Question:
Can employees work remotely three days per week?

Answer:
Yes, employees may work remotely up to 3 days per week if their role allows it and their manager approves.  
(Source: Remote Work Policy)

Sources retrieved:
- Remote Work Policy (score: 0.737)
- Remote Work Policy (score: 0.386)
- Company Vacation Policy (score: 0.358)


{'question': 'Can employees work remotely three days per week?',
 'answer': 'Yes, employees may work remotely up to 3 days per week if their role allows it and their manager approves.  \n(Source: Remote Work Policy)',
 'retrieved_chunks': [{'chunk_id': 'remote_work_policy_chunk_1',
   'doc_id': 'remote_work_policy',
   'title': 'Remote Work Policy',
   'text': 'Employees may work remotely up to 3 days per week if their role allows remote work and their manager approves the arrangement.\n\nRemote employees must be reachable during core collaboration hours from 10:00 to 16:00 local time.',
   'embedding': array([-0.01779175, -0.03411865, -0.00547409, ..., -0.00359154,
           0.00963593,  0.01638794], shape=(3072,), dtype=float32),
   'score': 0.737038254737854},
  {'chunk_id': 'remote_work_policy_chunk_2',
   'doc_id': 'remote_work_policy',
   'title': 'Remote Work Policy',
   'text': 'Employees who handle confidential data must use the company VPN and approved devices when working r

### Exercise 3

Modify the prompt so the model returns JSON in this shape:

```json
{
  "answer": "...",
  "sources": ["..."],
  "confidence": "low/medium/high"
}
```

Use `low`, `medium`, or `high` confidence based on how directly the context answers the question.

### Solution Exercise 3 - HIDE THIS CELL

This solution asks the model to return only valid JSON.

In [13]:
# SOLUTION CELL - HIDE BEFORE CLASS

import json


def build_json_rag_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context_blocks = []

    for index, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(
            f"Source {index}: {chunk['title']}\n{chunk['text']}"
        )

    context = '\n\n'.join(context_blocks)

    return f'''Use only the provided context to answer the question.

If the answer is not in the context, use exactly this answer value:
I don't know based on the provided documents.

Return only valid JSON in this exact shape:
{{
  "answer": "...",
  "sources": ["..."],
  "confidence": "low/medium/high"
}}

Context:
{context}

Question:
{question}'''


def answer_with_rag_json(question: str, top_k: int = 3) -> dict:
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_json_rag_prompt(question, retrieved)

    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
    )

    content = response.choices[0].message.content
    print(content)

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {'raw_response': content}


answer_with_rag_json('Can I upload confidential client data to public AI tools?', top_k=3)

```json
{
  "answer": "No, confidential client data must not be uploaded to public AI tools according to the Security Policy.",
  "sources": ["Security Policy"],
  "confidence": "high"
}
```


{'raw_response': '```json\n{\n  "answer": "No, confidential client data must not be uploaded to public AI tools according to the Security Policy.",\n  "sources": ["Security Policy"],\n  "confidence": "high"\n}\n```'}

### Exercise 4

Create a query that retrieves the wrong chunk.

Then try to improve the query or the chunking strategy.

Hint: vague queries are often harder for retrieval than specific queries.

### Solution Exercise 4 - HIDE THIS CELL

This solution compares a vague query with a clearer query, then tries smaller chunks.

In [14]:
# SOLUTION CELL - HIDE BEFORE CLASS

vague_query = 'Can I use the assistant for files?'
clear_query = 'Can I upload confidential client data to public AI tools?'

print('Vague query results')
print('=' * 80)
print_retrieved(retrieve(vague_query, top_k=3))

print('\n\nClearer query results')
print('=' * 80)
print_retrieved(retrieve(clear_query, top_k=3))

print('\n\nTrying smaller chunks with max_paragraphs=1')
print('=' * 80)

chunks = build_chunks_from_documents(documents, max_paragraphs=1)
for chunk in chunks:
    chunk['embedding'] = np.array(get_embedding(chunk['text']), dtype=np.float32)

print_retrieved(retrieve(clear_query, top_k=3))

print('\nDiscussion: Clearer queries and smaller chunks can improve retrieval, but they are not guaranteed to fix every case.')

Vague query results

Result 1 | score: 0.475 | source: Internal AI Assistant Policy
--------------------------------------------------------------------------------
The internal AI assistant may be used to summarize internal documents, draft non-client communications, brainstorm ideas, and explain technical concepts.

The internal AI assistant may process confidential data only when the tool is approved for that data classification and the user has permission to access the data.

Result 2 | score: 0.250 | source: Laptop Replacement Policy
--------------------------------------------------------------------------------
Accessories such as keyboards, mice, and laptop stands can be requested separately through the IT service portal.

Result 3 | score: 0.227 | source: Security Policy
--------------------------------------------------------------------------------
Confidential client data must not be uploaded to public AI tools, public file-sharing websites, or unapproved external services.

## 16. Final summary

- RAG connects LLMs to external knowledge.
- Retrieval selects relevant chunks from a knowledge base.
- Generation uses those chunks as context to produce a grounded answer.
- RAG is not magic. Quality depends on documents, chunking, embeddings, retrieval, prompts, and model behavior.
- In production, RAG requires evaluation, monitoring, access control, and human feedback.